# 🌾 Fine-Tuning de Gemma 4
Este notebook está impulsado por **Unsloth** para lograr un entrenamiento 2x más rápido y con bajo consumo de VRAM. Adapta el modelo Gemma 4 (E2B o E4B) para comprender imágenes de plantas y diagnosticar enfermedades basándose en el dataset de CDDMBench.


## 1. Configuración de Entorno
Instalamos Unsloth y otras dependencias clave para el entrenamiento de PyTorch y exportación de modelos móviles (Lite-RT).


In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!pip install -q -U uv  # Garantizar que el motor uv esté instalado en Kaggle
!uv pip install -qqq --system \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

# Utilidades extras
!uv pip install --system gdown

In [ ]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

## 2. Descarga del Dataset Multimodal
Descargamos las imágenes directamente de Google Drive a la máquina virtual y descomprimimos los JSON generados localmente. Esto nos evita problemas al subir archivos grandes mediante la interfaz web.


In [ ]:
%%bash
# 1. Descargando el ZIP (que contiene dataset/ y los .jsonl)
pip install -q gdown
gdown --id "1SKHuV5wlO11XPRk1CKJek_mK3hKvaJb9" -O gemma_4_dataset.zip --continue --no-cookies

# 2. Descomprimiendo en el directorio raíz de Colab
# Usamos -o para sobrescribir archivos existentes sin preguntar y evitar que el script se bloquee
unzip -q -o gemma_4_dataset.zip

# 3. Corrigiendo las rutas: el JSONL busca los archivos como 'Train/...',
# pero al inicio están en 'dataset/Train/...'. Movemos todo afuera:
if [ -d "dataset" ]; then
    mv dataset/* ./
    rm -rf dataset  # Borrar carpeta vacía
fi

rm -f gemma_4_dataset.zip # Limpieza

echo "✅ Dataset descomprimido y organizado. Carpetas Train, Val, Test y JSONLs listos."

## 3. Cargar Modelo Base Gemma 4
Cargamos el modelo base. Recomendamos la versión **E2B** para el prototipado inicial en Colab (T4 GPU - consume ~8GB de VRAM) y la **E4B** si tienes acceso a una A100. Todo carga en 4-bit para cuidar la memoria.


In [ ]:
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",      # Usa E4B-it si tienes > 16GB de VRAM
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
    max_seq_length = 2048,          # Límite seguro para diagnóstico de imágenes
)


In [ ]:
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    "gemma-4"
)

## 4. Configurar LoRA (Adaptadores Pequeños)
Aquí "conectamos" el aprendizaje neuronal a nuestro modelo. 
**NOTA CRUCIAL PARA VISIÓN:** Es obligatorio poner `finetune_vision_layers = True` para que el modelo aprenda fisiología vegetal, al igual que los módulos de atención.


In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,  # Clave para diagnóstico de enfermedades
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 64,                # Mayor rango (r=64) como pide el paper original (CDDM)
    lora_alpha = 64,       
    lora_dropout = 0.05,   # Dropout para evitar sobreajuste
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    target_modules = "all-linear",
)


## 5. Cargar Dataset Convertido
Cargamos los datos convertidos a la estructura requerida. También aplicaremos un **Subsampling** a 15,000 ejemplos para que el entrenamiento pueda completarse hoy mismo (toma unas ~5 horas).


In [ ]:
import json
import random

# 1. Cargar todas las líneas del formato JSONL compatible con entrenamiento
converted_dataset = []
with open("gemma_finetuning_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            converted_dataset.append(json.loads(line))

# 2. Mezclar el dataset aleatoriamente para evitar secuencias repetidas
random.seed(3407)
random.shuffle(converted_dataset)
subset_dataset = converted_dataset[:16000] # Limita al máximo deseado

print(f"✅ ¡Dataset listo! Usando {len(subset_dataset)} ejemplos para entrenamiento.")


## 6. Configurar Entrenamiento (SFTTrainer)
Configuración optimizada del entrenador Supervised Fine-Tuning. Está diseñada con memoria altamente eficiente (`batch=1`, `accum_steps=8`) y Learning Rate seguro.


In [ ]:
import transformers

# Parcheamos el comportamiento interno de Trainer
_old_trainer_init = transformers.Trainer.__init__

def _patched_init(self, *args, **kwargs):
    # Si alguna librería (como Unsloth) envía el argumento "tokenizer" bloqueado,
    # lo interceptamos y le cambiamos el nombre a escondidas a "processing_class"
    if "tokenizer" in kwargs:
        kwargs["processing_class"] = kwargs.pop("tokenizer")
    _old_trainer_init(self, *args, **kwargs)

transformers.Trainer.__init__ = _patched_init
print("✅ Parche Anti-Conflictos de Unsloth inyectado!")


In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    train_dataset = subset_dataset,
# Eliminado por conflicto de versión
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,  # Simula un batch de 8 para mayor estabilidad
        max_grad_norm = 0.3,
        warmup_steps = 50,
        
        # Estrategia de Épocas / LR adaptada a 15k ejemplos
        num_train_epochs = 2,             # Te tomará ~4-5 horas a 1 epoch de 15k en T4
        learning_rate = 2e-5,             
        
        logging_steps = 1,
        save_strategy = "steps",
        save_steps = 500,
        optim = "adamw_8bit",             # Optimización 8bit para cuidar RAM
        weight_decay = 0.1,               # Regularizador (ayuda contra overfitting)
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "output_bioguardia_gemma4",
        report_to = "none",

        # Requsitos OBLIGATORIOS para Unsloth Vision
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_seq_length = 2048,
    )
)


¡Iniciamos el aprendizaje! Es normal que el Loss empiece cerca de ~13 debido al gran vocabulario de Gemma-4. Debería estabilizarse pronto.


In [ ]:
import os
import torch

# Ver cuánta VRAM tenemos disponible
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Memoria reservada antes de entrenar: {start_gpu_memory} GB")

# 1. Buscar el último checkpoint guardado
checkpoint_dir = "output_bioguardia_gemma4"
last_checkpoint = None

if os.path.exists(checkpoint_dir):
    # Filtra solo las carpetas que empiezan con "checkpoint-"
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        # Ordena para obtener el checkpoint con el número más alto
        last_checkpoint = os.path.join(
            checkpoint_dir,
            sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        )
        print(f"🔥 Retomando entrenamiento desde: {last_checkpoint}")
    else:
        print("🌱 No hay checkpoints previos. Entrenando desde cero.")
else:
    print("🌱 No hay checkpoints previos. Entrenando desde cero.")

# 2. INICIAR ENTRENAMIENTO (Pasando el checkpoint si existe)
trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)

print("\n✅ ¡Entrenamiento completado exitosamente!")


## 7. Exportar y Guardar a Hugging Face
Finalmente, subimos al Hub de Hugging Face.

Te recomiendo utilizar un modelo de 16-bits para poder transformarlo luego en `.tflite` para la app (Vía LiteRT / MediaPipe).


In [ ]:
# Define tus credenciales
HF_TOKEN = ""
HF_REPO = "alvarog1318/gemma4-vision-crop-deseases"

print("--- EXPORTANDO LORA (Solo Pesos Extra, Ágil y Liviano) ---")
model.push_to_hub(HF_REPO + "_lora", token=HF_TOKEN)
processor.push_to_hub(HF_REPO + "_lora", token=HF_TOKEN)

import os, shutil
print("--- LIMPIANDO DISCO PARA FUSIÓN DE MODELOS --- (Evita OSError: [Errno 28])")
# 1. Borrar carpetas de imágenes
for folder in ["Train", "Val", "Test"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Eliminado: {folder}")

# 2. Borrar checkpoints antiguos pero dejar el último (por seguridad)
checkpoint_dir = "output_bioguardia_gemma4"
if os.path.exists(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        last_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        for ckpt in checkpoints:
            if ckpt != last_checkpoint:
                shutil.rmtree(os.path.join(checkpoint_dir, ckpt))
                print(f"Eliminado checkpoint antiguo: {ckpt}")

print("--- EXPORTANDO MODELO COMPLETO Y FUSIONADO (16 BITS - NECESARIO PARA ANDROID LITERT) ---")
# PRIORIDAD: Esto crea el formato estándar de hugging face listo para Litert_Torch
model.push_to_hub_merged(
    HF_REPO + "_16bit",
    processor,
    save_method = "merged_16bit",
    token = HF_TOKEN
)

print("--- EXPORTANDO GGUF (Optimizado para usar con Ollama / llama.cpp local) ---")
model.push_to_hub_gguf(
    HF_REPO + "_gguf",
    processor,
    quantization_method = "q4_k_m",
    token = HF_TOKEN
)

print("✅ TODO EXPORTADO ESTUPENDAMENTE.")
